# Same AOI across providers: MPC Landsat + CDSE Sentinel

Demonstrates the endpoint/signer abstraction: the **same** request shape pulls Landsat from Microsoft Planetary Computer (SAS URL signing) and Sentinel from Copernicus Data Space (S3 credentials) over one AOI. Only `data_source` (and CDSE keys) change. Offline cells show how the catalog resolves each; the pulls are recipes.

In [ ]:
from earthlens.stac import Catalog

cat = Catalog()
print("MPC Landsat id:", cat.resolve("planetary-computer", "landsat-c2-l2"))
print("MPC signer:", cat.get_endpoint("planetary-computer").signer)
print("CDSE signer:", cat.get_endpoint("cdse").signer)

## MPC Landsat (no account — SAS signing)

```python
from earthlens.earthlens import EarthLens

aoi = dict(lat_lim=[40.40, 40.45], lon_lim=[-3.72, -3.67],
           start="2024-06-01", end="2024-06-30")
landsat = EarthLens(data_source="planetary-computer",
                    variables={"landsat-c2-l2": ["red", "green", "blue"]},
                    path="out/aoi/landsat", max_items=1, **aoi).download()
```

## CDSE Sentinel (S3 keys)

```python
# export CDSE_S3_ACCESS_KEY / CDSE_S3_SECRET_KEY first (see Authentication)
sentinel = EarthLens(data_source="cdse",
                     variables={"sentinel-2-l2a": ["B04", "B03", "B02"]},
                     path="out/aoi/sentinel", max_items=1, **aoi).download()
```

Both write COGs over the identical AOI; the backend picked `MpcSasSigner` vs `CdseS3Signer` from the catalog. Reproject/stack them with pyramids to compare Landsat and Sentinel over the same scene.